In [1]:
!pip install 'powerlaw'
!pip install 'whittlehurst'

import bmll2 as b2
from bmll2 import reference, Security, NormalisedSecurity, SparkHelper, get_market_data, get_market_data_range, VenueMarketError, get_market_tables, save_spark_dataframe, load_spark_dataframe
b2.get_file('modules/auxiliary_functions.py')
b2.get_file('modules/auxiliary_functions_polars.py')
import auxiliary_functions as af
import auxiliary_functions_polars as af_pl

import random
import math
import pandas as pd
import polars as pl
import numpy as np
from pandas import StringDtype
from datetime import timedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import LogFormatterSciNotation

from statsmodels.sandbox.stats.runs import runstest_1samp 
import powerlaw
import itertools
import pylab
import scipy.stats
import scipy.fft
from scipy.optimize import curve_fit
from sklearn.model_selection import ParameterGrid

from statsmodels.tsa.stattools import acf
from whittlehurst import whittle, fbm, tdml
from scipy.stats import t


In [2]:
def process_stock_pl(stock_data, group = 'day', N = 4, participation_method = 'homogenous', alpha = 2):

    if group == 'day':
        stock_data = stock_data.with_columns(pl.col('Date').alias('group key'))

    elif group == 'month':
        stock_data = stock_data.with_columns(pl.col('Date').dt.truncate('1mo').alias('group key'))
    
    elif group == 'year':
        stock_data = stock_data.with_columns(pl.col('Date').dt.truncate('1y').alias('group key'))

    features_list_pl = []
    
    for trades in stock_data.partition_by('group key'):

        if trades.is_empty():
            continue

        trades = trades.sort(['DateTime', 'ExchangeSequenceNo'])
        
        f      = af_pl.trader_participation(N = N, method = participation_method, alpha = alpha, f_min = 1,
                                            f_max = trades.shape[0], seed = 1)
        c      = af_pl.cumulative_probs(f)
        output = af_pl.orders(N = N, trades = trades, cumulative_probs = c)
        
        assignments = []
        for trader_id, rows in enumerate(output):
            for r in rows:
                assignments.append((r, trader_id))

        assignments_df = pl.DataFrame(assignments, schema = ['row idx', 'trader id'], orient = 'row')

        trades = trades.with_row_index('row idx')
        trades = trades.join(assignments_df, on = 'row idx', how = 'inner')
        trades = trades.sort(['DateTime', 'ExchangeSequenceNo'])

        trades = trades.with_columns((pl.col('Trade Sign') != pl.col('Trade Sign').shift(1)).cast(pl.Int32)
            .cum_sum().over(['Date', 'trader id']).alias('metaorder id'))
        

        features = trades.group_by(['Date', 'trader id', 'metaorder id']
        ).agg([

            pl.col('Ticker').first(),
            pl.col('MIC').first(),
            pl.col('DateTime').first().alias('Start time'),
            pl.col('DateTime').last().alias('End time'),
            pl.col('Daily Volume').first(),

            pl.col('Intraday Volatility').first(),
            pl.col('20 AD volume').first(),
            pl.col('20 AD volatility').first(),

            pl.col('Trade Sign').first().alias('trade sign'),
            
            pl.col('Volume').sum().alias('volume traded'),
            pl.len().alias('number child orders'),
            
            pl.col('Mid-price before').first(),
            pl.col('Mid-price after').last(),
            pl.col('Mid-price after (delayed)').last(),
            
            pl.col('Mid-price before (1ms)').first(),
            pl.col('Mid-price after (1ms)').last(),
            pl.col('Mid-price after (1ms)(delayed)').last()
        ])

        features = features.filter(pl.col('number child orders') > 1)

        features = features.with_columns([
            ((pl.col('Mid-price after').log() - pl.col('Mid-price before').log())
             * pl.col('trade sign')).alias('impact'),
        
            ((pl.col('Mid-price after (delayed)').log() - pl.col('Mid-price before').log())
             * pl.col('trade sign')).alias('impact (delayed)'),
        
            ((pl.col('Mid-price after (1ms)').log() - pl.col('Mid-price before (1ms)').log())
             * pl.col('trade sign')).alias('impact (1ms)'),
        
            ((pl.col('Mid-price after (1ms)(delayed)').log() - pl.col('Mid-price before (1ms)').log())
             * pl.col('trade sign')).alias('impact (1ms)(delayed)'), 
            
            (((pl.col('End time') - pl.col('Start time')).dt.total_microseconds() / 6e7).alias('duration (min)'))
        ])
        
        features_list_pl.append(features)

    if features_list_pl:
        return pl.concat(features_list_pl, how = 'vertical', rechunk = True)
    else:
        return pl.DataFrame()


def concave_profile_pl(stock_data, group = 'day', N = 4, participation_method = 'homogenous', alpha = 2):

    if group == 'day':
        stock_data = stock_data.with_columns(pl.col('Date').alias('group key'))

    elif group == 'month':
        stock_data = stock_data.with_columns(pl.col('Date').dt.truncate('1mo').alias('group key'))
    
    elif group == 'year':
        stock_data = stock_data.with_columns(pl.col('Date').dt.truncate('1y').alias('group key'))

    impact_profile = []
    
    for trades in stock_data.partition_by('group key'):
        
        if trades.is_empty():
            continue

        trades = trades.sort(['DateTime', 'ExchangeSequenceNo'])
        
        f      = af_pl.trader_participation(N = N, method = participation_method, alpha = alpha, f_min = 1,
                                            f_max = trades.shape[0], seed = 1)
        c      = af_pl.cumulative_probs(f)
        output = af_pl.orders(N = N, trades = trades, cumulative_probs = c)
        
        assignments = []
        for trader_id, rows in enumerate(output):
            for r in rows:
                assignments.append((r, trader_id))

        assignments_df = pl.DataFrame(assignments, schema = ['row idx', 'trader id'], orient = 'row')

        trades = trades.with_row_index('row idx')
        trades = trades.join(assignments_df, on = 'row idx', how = 'inner')
        trades = trades.sort(['DateTime', 'ExchangeSequenceNo'])

        trades = trades.with_columns((pl.col('Trade Sign') != pl.col('Trade Sign').shift(1)).cast(pl.Int32)
            .cum_sum().over(['Date', 'trader id']).alias('metaorder id'))
            
        grouping_cols = ['Date', 'trader id', 'metaorder id']
        features = trades.with_columns([
            
            pl.col('Ticker'),
            pl.col('MIC'),
            pl.col('DateTime').first().alias('Start time'),
            pl.col('DateTime').last().alias('End time'),
            
            pl.col('Daily Volume').first().over(grouping_cols),
            pl.col('Intraday Volatility').first().over(grouping_cols),
            pl.col('20 AD volume').first().over(grouping_cols),
            pl.col('20 AD volatility').first().over(grouping_cols),

            pl.col('Trade Sign').alias('trade sign'),

            pl.col('Volume').alias('volume'),
            pl.col('Volume').sum().over(grouping_cols).alias('volume traded'),
            (pl.col('Volume').cum_sum().over(grouping_cols) / pl.col('Volume').sum().over(grouping_cols)).alias('phi'),
            pl.len().over(grouping_cols).alias('number child orders'),
            
            pl.col('Mid-price before'),
            pl.col('Mid-price after'),

            (pl.col('Trade Sign').first().over(grouping_cols) * (pl.col('Mid-price after').last().log().over(grouping_cols) - pl.col('Mid-price before').first().log().over(grouping_cols))).alias('metaorder impact')
        ])
        
        #features = features.filter(pl.col('number child orders') > 4)
        
        features = features.with_columns([
            (pl.col('trade sign') * (pl.col('Mid-price after').log() - pl.col('Mid-price before').first().log().over(grouping_cols)) / (pl.col('20 AD volatility') * (pl.col('volume traded') / pl.col('20 AD volume')).sqrt())).alias('scaled impact'),
            (((pl.col('End time') - pl.col('Start time')).dt.total_microseconds() / 6e7).alias('duration (min)'))
        ])        
        features = features.select('MIC', 'Ticker', 'Date', 'DateTime', '20 AD volume', '20 AD volatility', 'trade sign',
                                   'trader id', 'metaorder id', 'number child orders', 'volume', 'volume traded',
                                   'phi', 'scaled impact', 'metaorder impact')
        
        impact_profile.append(features)

    if  impact_profile:
        return pl.concat(impact_profile, how = 'vertical', rechunk = True)
    else:
        return pl.DataFrame()

def decay_data_pl(stock_data, group = 'day', N = 4, participation_method = 'homogenous', alpha = 2):

    ticker = stock_data['Ticker'][0]
    if group == 'day':
        stock_data = stock_data.with_columns(pl.col('Date').alias('group key'))

    elif group == 'month':
        stock_data = stock_data.with_columns(pl.col('Date').dt.truncate('1mo').alias('group key'))
    
    elif group == 'year':
        stock_data = stock_data.with_columns(pl.col('Date').dt.truncate('1y').alias('group key'))

    features_list_pl = []
    
    for trades in stock_data.partition_by('group key'):

        if trades.is_empty():
            continue

        trades = trades.sort(['DateTime', 'ExchangeSequenceNo'])

        f      = af_pl.trader_participation(N = N, method = participation_method, alpha = alpha, f_min = 1,
                                            f_max = trades.shape[0], seed = 1)
        c      = af_pl.cumulative_probs(f)
        output = af_pl.orders(N = N, trades = trades, cumulative_probs = c)
        
        assignments = []
        for trader_id, rows in enumerate(output):
            for r in rows:
                assignments.append((r, trader_id))

        assignments_df = pl.DataFrame(assignments, schema = ['row idx', 'trader id'], orient = 'row')

        trades = trades.sort(['Date', 'DateTime', 'ExchangeSequenceNo'])
        trades = trades.with_row_index('row idx')
        trades = trades.join(assignments_df, on = 'row idx', how = 'inner')
        trades = trades.sort(['DateTime', 'ExchangeSequenceNo'])
        
        trades = trades.with_columns((pl.col('Trade Sign') != pl.col('Trade Sign').shift(1)).cast(pl.Int32)
            .cum_sum().over(['Date', 'trader id']).alias('metaorder id'))

        features = trades.group_by(['Date', 'trader id', 'metaorder id']
        ).agg([

            pl.col('Ticker').first(),
            pl.col('MIC').first(),
            pl.col('DateTime').first().alias('Start time'),
            pl.col('DateTime').last().alias('End time'),

            pl.col('Daily Volume').first(),
            pl.col('Intraday Volatility').first(),
            pl.col('20 AD volume').first(),
            pl.col('20 AD volatility').first(),

            pl.col('Trade Sign').first().alias('trade sign'),
            
            pl.col('Volume').sum().alias('volume traded'),
            pl.len().alias('number child orders'),
            
            pl.col('Mid-price before').first(),
            pl.col('Mid-price after').last(),
        ])

        features = features.filter(pl.col('number child orders') > 4)

        features = features.with_columns([
            ((pl.col('Mid-price after').log() - pl.col('Mid-price before').log())
             * pl.col('trade sign')).alias('impact'),

            (((pl.col('End time') - pl.col('Start time')).dt.total_microseconds()).alias('duration (ms)'))
        ])
        
        features_list_pl.append(features)

    if features_list_pl:
        metaorder_data = pl.concat(features_list_pl, how = 'vertical', rechunk = True)

        stock_mid_prices = get_market_data_range('XJSE', start_date = '2023-01-03', end_date = '2025-12-31',
                                       table_name = 'l1', ticker = ticker, df_engine = 'polars',
                                      columns = ['Ticker', 'TradeDate', 'LocalTimestamp', 'BidPrice1', 'BidQuantity1', 'AskPrice1', 'AskQuantity1',
                                                 'ExchangeSequenceNo', 'MarketState'])
        stock_mid_prices = stock_mid_prices.filter(pl.col('MarketState') == 'CONTINUOUS_TRADING')
        stock_mid_prices = stock_mid_prices.sort(['LocalTimestamp', 'ExchangeSequenceNo'])
        stock_mid_prices = stock_mid_prices.with_columns(((pl.col('BidPrice1') + pl.col('AskPrice1')) / 2).alias('Mid-price'))
        stock_mid_prices = stock_mid_prices.with_columns(pl.col("LocalTimestamp").dt.replace_time_zone(None))
        stock_mid_prices = stock_mid_prices.sort('LocalTimestamp')
        
        z_max  = 2
        z_grid = np.linspace(1, z_max, 100)
        
        meta = metaorder_data.filter(pl.col('number child orders') > 4)
        meta = meta.sort('Start time')
        
        expanded = meta.with_row_index('meta id').join(pl.DataFrame({'z': z_grid}), how = 'cross')
        expanded = expanded.with_columns((pl.col('Start time') + pl.duration(microseconds = (pl.col('z') * (pl.col('End time') - pl.col('Start time')).dt.total_microseconds()).cast(pl.Int64))).alias('query time'))
        expanded = expanded.sort('query time')
        
        merged = expanded.join_asof(stock_mid_prices, left_on = 'query time', right_on = 'LocalTimestamp', strategy = 'backward')
        merged = merged.filter(pl.col('LocalTimestamp') >= pl.col('End time'))
        merged = merged.with_columns([pl.col('Mid-price before').alias('P0'), pl.col('Mid-price').alias('Pz')])
        merged = merged.with_columns((pl.col('trade sign') * (pl.col('Pz').log() - pl.col('P0').log())).alias('impact'))
        merged = merged.with_columns((pl.col('impact') / (pl.col('20 AD volatility') * (pl.col('volume traded') / pl.col('20 AD volume')).sqrt())).alias('scaled impact'))
        
        decay_df = merged.select(['Date', 'z', 'impact', 'scaled impact'])

        return decay_df
        
    else:
        return pl.DataFrame()
        

In [3]:
def one_sided_runs_test(sample: list, runs_correction = 0):

    if len(sample) < 2:
        return None

    if sum(sample) < 100:
        return None

    N   = sum(sample)
    N_a = sum(sample[::2])
    N_b = sum(sample[1::2])

    mu   = ((2 * N_a * N_b) / N) + 1
    runs = len(sample)

    R_corrected = runs + runs_correction
    
    sigma = np.sqrt((2 * N_a * N_b * (2 * N_a * N_b - N)) / (N ** 2 * (N - 1)))

    z = (R_corrected - mu) / sigma
    p_value = scipy.stats.norm.cdf(z)

    return (z, p_value, runs)

def extract_ST_run_lengths(stock_data, group = 'year', N = 100, trader_distribution = 'power',
                           alpha = 2, p_threshold = 0.01):

    stock_data = stock_data.sort(['DateTime', 'ExchangeSequenceNo'], descending = False)

    if group == 'day':
        stock_data = stock_data.with_columns(pl.col('Date').alias('group key'))

    elif group == 'month':
        stock_data = stock_data.with_columns(pl.col('Date').dt.truncate('1mo').alias('group key'))
    
    elif group == 'year':
        stock_data = stock_data.with_columns(pl.col('Date').dt.truncate('1y').alias('group key'))

    run_lengths = []
    
    for trades in stock_data.partition_by('group key'):

        if trades.is_empty():
            continue
            
        trades = trades.sort(['Date', 'DateTime', 'ExchangeSequenceNo'])
        f = af_pl.trader_participation(N = N, method = trader_distribution, alpha =  alpha, f_min = 1, 
                                       f_max = trades.shape[0], seed = 1)
        c = af_pl.cumulative_probs(f)
        output = af_pl.orders(N = N, trades = trades, cumulative_probs = c)
    
        assignments = []
        for trader_id, rows in enumerate(output):
            for r in rows:
                assignments.append((r, trader_id))
                
        assignments_df = pl.DataFrame(assignments, schema = ['row idx', 'trader id'], orient = 'row')
    
        trades = trades.with_row_index('row idx')
        trades = trades.join(assignments_df, on = 'row idx', how = 'inner')
    
        trades = trades.with_columns((pl.col('Trade Sign') != pl.col('Trade Sign').shift(1)).cast(pl.Int32)
            .cum_sum().over(['Date', 'trader id']).alias('metaorder id'))
    
        grouping_cols = ['trader id', 'metaorder id']
    
        features = trades.with_columns([pl.col('DateTime').diff().over('trader id').alias('gap'), 
                                        (pl.col('Trade Sign') == pl.col('Trade Sign').shift(1)).over('trader id').fill_null(False).alias('same sign')])
        features = features.with_columns(((pl.col('gap') > timedelta(days = 1)) & pl.col('same sign')).fill_null(False).cast(pl.Int64).alias('day break'))
        features = features.with_columns((pl.col('Trade Sign') != pl.col('Trade Sign').shift(1)).over('trader id').fill_null(True).cast(pl.Int64).cum_sum().alias('run id'))
        run_lengths_df = features.group_by(['trader id', 'run id']).agg(pl.col('run id').len().alias('L'), pl.col('day break').max().alias('day break'))
        run_lengths_df = run_lengths_df.with_columns(pl.col('day break').sum().over('trader id').alias('day breaks'))
        run_lengths_df = run_lengths_df.select(['trader id', 'L', 'day breaks'])
        runs_per_trader = run_lengths_df.group_by('trader id').agg([pl.col('L'),pl.col('day breaks').first()])
        runs_per_trader = runs_per_trader.sort('trader id')
        
        for row in runs_per_trader.iter_rows(named = True):
        
            runs = row['L']
            day_breaks = row['day breaks']
        
            if len(runs) < 2:
                continue

            if sum(runs) < 100:
                continue

            z, p_val, R = one_sided_runs_test(runs, runs_correction = day_breaks)
            
            if p_val <= p_threshold:
                run_lengths.extend(runs)

    return run_lengths
    
def acf_fft(signs: pl.Series, max_lag):

    signs   = np.asarray(signs)
    N       = len(signs)
    max_lag = min(max_lag, N - 1)

    nfft = 1 << (2 * N - 1).bit_length()

    f   = np.fft.fft(signs, nfft)
    acf = np.fft.ifft(f * np.conjugate(f)).real
    acf = acf[:max_lag + 1]

    norm = np.arange(N, N - max_lag - 1, -1)

    C    = acf[1:] / norm[1:]
    taus = np.arange(1, max_lag + 1)

    return taus, C

def select_powerlaw_range_fast(taus, C, min_points = 10000, confidence_percent = 0.05):

    log_tau = np.log(taus)
    log_C   = np.log(C)

    N = len(log_tau)

    # cumulative sums
    # x is the log taus
    # y is the log C
    Sx  = np.cumsum(log_tau)
    Sy  = np.cumsum(log_C)
    Sxx = np.cumsum(log_tau ** 2)
    Syy = np.cumsum(log_C ** 2)
    Sxy = np.cumsum(log_tau * log_C)

    best_r2 = -np.inf
    best_i  = 0
    best_slope = 0

    j = N - 1  # fixed right endpoint (tail fit)

    for i in range(N - min_points):
        
        n = j - i

        # gonna be working backwards here. We already precomputed the cumulative sum of the entire series we just need to subtract
        # the part thats no longer in it. It starts off on the largest window and makes it smaller each time. With this method we fix
        # the left endpoint of the window so that we only need to do a O(N) search and not a O(N^2) search. 
        sum_x  = Sx[j - 1]  - (Sx[i - 1]  if i > 0 else 0)
        sum_y  = Sy[j - 1]  - (Sy[i - 1]  if i > 0 else 0)
        sum_xx = Sxx[j - 1] - (Sxx[i - 1] if i > 0 else 0)
        sum_yy = Syy[j - 1] - (Syy[i - 1] if i > 0 else 0)
        sum_xy = Sxy[j - 1] - (Sxy[i - 1] if i > 0 else 0)

        denom = n * sum_xx - sum_x ** 2
        if denom == 0:
            continue

        slope = ((n * sum_xy) - (sum_x * sum_y)) / denom
        intercept = (sum_y - (slope * sum_x)) / n

        ss_tot = sum_yy - (sum_y ** 2) / n
        ss_res = sum_yy + (slope ** 2) * sum_xx + (n * intercept ** 2) - (2 * slope * sum_xy) - (2 * intercept * sum_y) + (2 * slope * intercept * sum_x)

        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else -np.inf

        if r2 > best_r2:
            best_r2 = r2
            best_i  = i
            best_slope = slope

    return best_i, N - 1, best_slope

def powerlaw_fit(tau, C_0, gamma):
    
    return C_0 * (tau ** (- gamma))

def gamma_NLLS(taus, C):

    x = np.log(taus)
    y = np.log(C)

    slope, intercept = np.polyfit(x, y, 1)
    gamma            = -1 * slope

    return intercept, gamma

def gamma_PSD(C):

    max_lag = len(C)
    PSD     = np.fft.fft(C)
    freqs   = np.fft.fftfreq(max_lag)

    mask  = freqs > 0
    PSD   = PSD[mask]
    freqs = freqs[mask]

    cut       = int(0.15 * len(freqs))
    log_PSD   = np.log(PSD[: cut])
    log_freqs = np.log(freqs[: cut])
    
    slope, intercept  = np.polyfit(log_freqs, log_PSD, 1)
    gamma_psd = slope + 1
    
    return np.real(intercept), np.real(gamma_psd)

def estimate_gamma(taus, C, min_points, nbins, trade_signs, confidence_percent = 0.05):

    z_score = scipy.stats.norm.ppf(1 - confidence_percent)
    
    mask              = (C > (z_score / np.sqrt(len(C)))) & np.isfinite(C)
    cut = np.argmax(~mask)
    if cut == 0 and mask[0]:
        # mask is all True → keep everything
        taus, C_truncated = taus, C
    else:
        taus, C_truncated = taus[:cut], C[:cut]
    #taus, C_truncated = taus[:np.argmax(~mask)], C[:np.argmax(~mask)] # the tilde flips it so that where C < ... it says true. 
    # Then argmax returns the largest value where its true.
    # The mask still has the C values in it so it is checking the C values and not just the true false stuff
    #mask = (C > 0) & (np.isfinite(C))
    #taus = taus[mask]
    #C    = C[mask]
    
    i, j, slope     = select_powerlaw_range_fast(taus, C_truncated, min_points)
    gamma_nlls_init = -1 * slope

    tau_min = taus[i]
    tau_max = taus[j]

    mask_fit    = (taus >= tau_min) & (taus <= tau_max)
    taus        = taus[mask_fit]
    C_truncated = C_truncated[mask_fit]

    C_0_init           = C[0] if len(C) > 0 else 1.0
    C_nlls, gamma_nlls = gamma_NLLS(taus, C_truncated)

    H                = whittle(trade_signs)
    gamma_whittle    = 2 - 2 * H
    C_psd, gamma_psd = gamma_PSD(C)

    return {'intercept' : C_nlls, 'gamma (nlls)' : gamma_nlls, 'gamma (whittle)': gamma_whittle, 'gamma (PSD)': gamma_psd}

def estimate_alpha(run_lengths):
    
    fit = powerlaw.Fit(run_lengths, discrete = True, verbose = False)
    
    # The powerlaw package gives alpha for CCDF P_>(L) ~ L^(-alpha)
    # For PDF P(L) ~ L^(-alpha-1), we have the same alpha
    alpha = fit.alpha - 1  # Adjust for PDF vs CCDF convention
    xmin  = fit.xmin
    
    return alpha, xmin
        

In [4]:
""""
def metaorders_stylised_facts(stock_data, N, alpha_tp, participation_method = 'power'):

    years_col         = []
    ticker_col        = []
    xmin_col          = []
    alpha_col         = []
    gamma_nlls_col    = []
    gamma_psd_col     = []
    sql_est_col       = []
    sql_var_col       = []
    sql_interval_half_width_col  = []
    phi_est_col                  = []
    phi_var_col                  = []
    phi_interval_half_width_col  = []
    beta_est_col                 = []
    beta_var_col                 = []
    beta_interval_half_width_col = []

    concave_data_all   = concave_profile_pl(stock_data, group = 'year', N = N,
                                            participation_method = participation_method,
                                            alpha = alpha_tp)
    concave_data_all   = concave_data_all.with_columns(pl.col('Date').dt.truncate('1y').dt.year().alias('Year'))
    
    decay_data_all     = decay_data_pl(stock_data, group = 'year', N = N,
                                       participation_method = participation_method,
                                       alpha = alpha_tp)
    decay_data_all     = decay_data_all.with_columns(pl.col('Date').dt.truncate('1y').dt.year().alias('Year'))
    
    metaorder_data_all = concave_data_all.filter((pl.col('phi') == 1) & (pl.col('number child orders') > 1))
    
    concave_data_all   = concave_data_all.filter(pl.col('number child orders') > 4)

    stock_data = stock_data.with_columns(pl.col('Date').dt.truncate('1y').alias('Year'))
    
    for df_year in stock_data.partition_by('Year'):
    
        year   = df_year['Year'].dt.year()[0]
        ticker = df_year['Ticker'][0]
        
        if df_year.shape[0] < 1000:
            continue

        concave_data   = concave_data_all.filter(pl.col('Year') == year)
        decay_data     = decay_data_all.filter(pl.col('Year') == year)
        metaorder_data = metaorder_data_all.filter(pl.col('Year') == year)
    
        if concave_data.is_empty() or decay_data.is_empty() or metaorder_data.is_empty():
            return pl.DataFrame()

        ### SQL ###
        n_intervals = 40
        
        df = metaorder_data.with_columns((pl.col('volume traded') / pl.col('20 AD volume')).alias('x_meta'),
                                         (pl.col('metaorder impact') / pl.col('20 AD volatility')).alias('y_meta'))
        
        x_min = df.select(pl.col('x_meta').min()).item()
        x_max = df.select(pl.col('x_meta').max()).item()
        
        bins = np.logspace(np.log10(x_min), np.log10(x_max), n_intervals)
        df   = df.with_columns([pl.Series('bin', np.digitize(df['x_meta'].to_numpy(), bins) - 1)])
        
        all_bins = pl.DataFrame({'bin': np.arange(len(bins) - 1)})
        binned   = df.group_by('bin').agg(pl.col('y_meta').mean().alias('y_mean')).sort('bin')
        binned   = all_bins.join(binned, on = 'bin', how = 'left').sort('bin')
        
        bin_centers = (bins[:-1] + bins[1:]) / 2
        binned      = binned.with_columns(x_center = bin_centers)
        
        alpha  = 0.5
        binned = binned.with_columns((alpha * pl.col('x_center').sqrt()).alias('sqrt_law'))

        def sql_model(x, exp, Y):
            return Y * x ** exp
            
        y = np.array(binned['y_mean'])
        x = np.array(binned['x_center'])
        
        mask       = (x > 0) & np.isfinite(y)
        x_fit      = x[mask]
        impact_fit = y[mask]
        
        if len(x_fit) < 10 or len(impact_fit) < 10:
            return pl.DataFrame()
                    
        try:
            params, covariance = curve_fit(sql_model, x_fit, impact_fit, p0 = [0.5, 0.5], maxfev = 100000)
    
        except RuntimeError:
            return pl.DataFrame()

        exp_est = params[0]
        
        n   = len(impact_fit)
        p   = len(params)          # number of parameters (2 here)
        dof = n - p                # degrees of freedom
        
        alpha = 0.05               # 95% CI
        tval  = t.ppf(1 - alpha / 2, dof)
        se    = np.sqrt(np.diag(covariance))
        exp_interval_half_width = tval * se[0]
        exp_var = covariance[0, 0]
        
        ###
        
        ### Execution profile ###
        
        bins = np.linspace(0, 1, 51)
        bin_mids = (bins[:-1] + bins[1:]) / 2
        
        df        = concave_data.with_columns(pl.col('phi').cut(bins).alias('bin'))
        binned    = df.group_by('bin').agg(pl.col('scaled impact').mean().alias('scaled impact (binned)'))
        full_bins = pl.DataFrame({'phi': bin_mids}).with_columns(pl.col('phi').cut(bins).alias('bin'))
        df        = full_bins.join(binned, on = 'bin', how = 'left').sort('phi')
        
        scaled_impact_binned = df['scaled impact (binned)']
        if len(scaled_impact_binned) < 10:
            return pl.DataFrame()
            
        scaled_impact_binned = pl.concat([pl.Series([0.0]), scaled_impact_binned])
        phi                  = np.insert(bin_mids, 0, 0)
        
        def execution_profile_model(phi, exponent, Y):
            return  Y * (phi ** exponent)
        
        phi    = np.array(phi)
        impact = np.array(scaled_impact_binned)
        
        mask       = (phi > 0) & np.isfinite(impact)
        phi_fit    = phi[mask]
        impact_fit = impact[mask]
    
        if len(phi_fit) < 10 or len(impact_fit) < 10:
            return pl.DataFrame()
        
        try:
            params, covariance = curve_fit(execution_profile_model, phi_fit, impact_fit, p0 = [0.5, 1], maxfev = 100000)
    
        except RuntimeError:
            return pl.DataFrame()
            
        phi_est = params[0]
        
        n   = len(impact_fit)
        p   = len(params)          # number of parameters (2 here)
        dof = n - p                # degrees of freedom
        
        alpha = 0.05               # 95% CI
        tval  = t.ppf(1 - alpha / 2, dof)
        se    = np.sqrt(np.diag(covariance))
        phi_interval_half_width = tval * se[0]    
        phi_var = covariance[0, 0]
        
        ###
        
        ### Decay profile ###
        
        bins = np.linspace(1, 2, 51)
        bin_mids = (bins[:-1] + bins[1:]) / 2
        
        df        = decay_data.with_columns(pl.col('z').cut(bins).alias('bin'))
        binned    = df.group_by('bin').agg(pl.col('scaled impact').mean().alias('scaled impact (binned)')).sort('bin')
        full_bins = pl.DataFrame({'z': bin_mids}).with_columns(pl.col('z').cut(bins).alias('bin'))
        df        = full_bins.join(binned, on = 'bin', how = 'left').sort('z')
       
        z         = np.array(bin_mids)
        impact    = np.array(df['scaled impact (binned)'])
        
        def decay_model(z, beta, Y):
            return  Y * (z ** (1 - beta) - (z - 1) ** (1 - beta))
        
        mask       = (z > 1) & np.isfinite(impact)
        z_fit      = z[mask]
        impact_fit = impact[mask]
    
        if len(z_fit) < 10 or len(impact_fit) < 10:
            return pl.DataFrame()
    
        try:
            params, covariance = curve_fit(decay_model, z_fit, impact_fit, p0 = [0.2, 1], maxfev = 100000)
    
        except RuntimeError:
            return pl.DataFrame()    
        
        beta_est = params[0]
    
        n   = len(z_fit)
        p   = len(params)          # number of parameters (2 here)
        dof = n - p                # degrees of freedom
        
        alpha = 0.05               # 95% CI
        tval  = t.ppf(1 - alpha / 2, dof)
        se    = np.sqrt(np.diag(covariance))
        beta_interval_half_width = tval * se[0]
        beta_var = covariance[0, 0] 
        
        L     = extract_ST_run_lengths(df_year, group = 'year', N = N, trader_distribution = participation_method,
                                       alpha = alpha_tp)
        signs = df_year['Trade Sign']
        
        if len(L) == 0:
            continue
    
        L     = pl.DataFrame({'L' : L})
        signs = pl.DataFrame({'Trade Sign' : signs})
        
        taus, C = acf_fft(signs['Trade Sign'], max_lag = 100000)
        out     = estimate_gamma(taus, C, min_points = 100, nbins = 100, trade_signs = signs['Trade Sign'])
        
        alpha, xmin = estimate_alpha(L['L'])
        
        years_col.append(year)
        ticker_col.append(ticker)
        xmin_col.append(xmin)
        alpha_col.append(alpha)
        gamma_nlls_col.append(out['gamma (nlls)'])
        gamma_psd_col.append(out['gamma (PSD)'])
        sql_est_col.append(exp_est)
        sql_var_col.append(exp_var)
        sql_interval_half_width_col.append(exp_interval_half_width)
        phi_est_col.append(phi_est)
        phi_var_col.append(phi_var)
        phi_interval_half_width_col.append(phi_interval_half_width)
        beta_est_col.append(beta_est)
        beta_var_col.append(beta_var)
        beta_interval_half_width_col.append(beta_interval_half_width)

    data_lists = [years_col, ticker_col, xmin_col, alpha_col, gamma_nlls_col, gamma_psd_col, 
                  phi_est_col, phi_interval_half_width_col, beta_est_col, beta_interval_half_width_col]
    
    if any(not x for x in data_lists):
        return pl.DataFrame()
        
    alpha_gamma_df = pl.DataFrame({
            'year'                    : years_col,
            'ticker'                  : ticker_col,
            'N'                       : N,
            'delta'                   : alpha_tp,
            'xmin'                    : xmin_col,
            'alpha'                   : alpha_col,
            'gamma (nlls)'            : gamma_nlls_col,
            'gamma (psd)'             : gamma_psd_col,
            'impact curve exponent'   : sql_est_col,
            'impact curve exponent var' : sql_var_col,
            'impact curve half width' : sql_interval_half_width_col,
            'phi exponent'            : phi_est_col, 
            'phi exponent var'        : phi_var_col,
            'phi exponent half width' : phi_interval_half_width_col,
            'beta'                    : beta_est_col,
            'beta var'                : beta_var_col,
            'beta half width'         : beta_interval_half_width_col}, 
             schema = {
            'year'                    : pl.Int64,
            'ticker'                  : pl.Utf8,
            'N'                       : pl.Int64,
            'delta'                   : pl.Float64,
            'xmin'                    : pl.Float64,
            'alpha'                   : pl.Float64,
            'gamma (nlls)'            : pl.Float64,
            'gamma (psd)'             : pl.Float64,
            'impact curve exponent'   : pl.Float64,
            'impact curve exponent var' : pl.Float64,
            'impact curve half width' : pl.Float64,
            'phi exponent'            : pl.Float64,
            'phi exponent var'        : pl.Float64,
            'phi exponent half width' : pl.Float64,
            'beta'                    : pl.Float64,
            'beta var'                : pl.Float64,
            'beta half width'         : pl.Float64})
    
    return alpha_gamma_df
""""

In [4]:
def metaorders_stylised_facts(stock_data, N, alpha_tp, participation_method = 'homogenous'):

    years_col         = []
    ticker_col        = []
    xmin_col          = []
    alpha_col         = []
    gamma_nlls_col    = []
    gamma_psd_col     = []
    sql_est_col       = []
    sql_var_col       = []
    sql_interval_half_width_col  = []
    phi_est_col                  = []
    phi_var_col                  = []
    phi_interval_half_width_col  = []
    beta_est_col                 = []
    beta_var_col                 = []
    beta_interval_half_width_col = []

    concave_data_all   = concave_profile_pl(stock_data, group = 'year', N = N,
                                            participation_method = participation_method,
                                            alpha = alpha_tp)
    concave_data_all   = concave_data_all.with_columns(pl.col('Date').dt.truncate('1y').dt.year().alias('Year'))
    
    decay_data_all     = decay_data_pl(stock_data, group = 'year', N = N,
                                       participation_method = participation_method,
                                       alpha = alpha_tp)
    decay_data_all     = decay_data_all.with_columns(pl.col('Date').dt.truncate('1y').dt.year().alias('Year'))
    
    metaorder_data_all = concave_data_all.filter((pl.col('phi') == 1) & (pl.col('number child orders') > 1))
    
    concave_data_all   = concave_data_all.filter(pl.col('number child orders') > 4)

    stock_data = stock_data.with_columns(pl.col('Date').dt.truncate('1y').alias('Year'))
    
    for df_year in stock_data.partition_by('Year'):
    
        year   = df_year['Year'].dt.year()[0]
        ticker = df_year['Ticker'][0]
        
        if df_year.shape[0] < 1000:
            continue

        concave_data   = concave_data_all.filter(pl.col('Year') == year)
        decay_data     = decay_data_all.filter(pl.col('Year') == year)
        metaorder_data = metaorder_data_all.filter(pl.col('Year') == year)
    
        if concave_data.is_empty() or decay_data.is_empty() or metaorder_data.is_empty():
            return pl.DataFrame()

        ### SQL ###
        n_intervals = 40
        
        df = metaorder_data.with_columns((pl.col('volume traded') / pl.col('20 AD volume')).alias('x_meta'),
                                         (pl.col('metaorder impact') / pl.col('20 AD volatility')).alias('y_meta'))
        
        x_min = df.select(pl.col('x_meta').min()).item()
        x_max = df.select(pl.col('x_meta').max()).item()
        
        bins = np.logspace(np.log10(x_min), np.log10(x_max), n_intervals)
        df   = df.with_columns([pl.Series('bin', np.digitize(df['x_meta'].to_numpy(), bins) - 1)])
        
        all_bins = pl.DataFrame({'bin': np.arange(len(bins) - 1)})
        binned   = df.group_by('bin').agg(pl.col('y_meta').mean().alias('y_mean')).sort('bin')
        binned   = all_bins.join(binned, on = 'bin', how = 'left').sort('bin')
        
        bin_centers = (bins[:-1] + bins[1:]) / 2
        binned      = binned.with_columns(x_center = bin_centers)
        
        alpha  = 0.5
        binned = binned.with_columns((alpha * pl.col('x_center').sqrt()).alias('sqrt_law'))

        def sql_model(x, exp, Y):
            return Y * x ** exp
            
        y = np.array(binned['y_mean'])
        x = np.array(binned['x_center'])
        
        mask       = (x > 0) & np.isfinite(y)
        x_fit      = x[mask]
        impact_fit = y[mask]
        
        if len(x_fit) < 10 or len(impact_fit) < 10:
            return pl.DataFrame()
                    
        try:
            params, covariance = curve_fit(sql_model, x_fit, impact_fit, p0 = [0.5, 0.5], maxfev = 100000)
    
        except RuntimeError:
            return pl.DataFrame()

        exp_est = params[0]
        
        n   = len(impact_fit)
        p   = len(params)          # number of parameters (2 here)
        dof = n - p                # degrees of freedom
        
        alpha = 0.05               # 95% CI
        tval  = t.ppf(1 - alpha / 2, dof)
        se    = np.sqrt(np.diag(covariance))
        exp_interval_half_width = tval * se[0]
        exp_var = covariance[0, 0]
        
        ###
        
        ### Execution profile ###
        
        bins = np.linspace(0, 1, 51)
        bin_mids = (bins[:-1] + bins[1:]) / 2
        
        df        = concave_data.with_columns(pl.col('phi').cut(bins).alias('bin'))
        binned    = df.group_by('bin').agg(pl.col('scaled impact').mean().alias('scaled impact (binned)'))
        full_bins = pl.DataFrame({'phi': bin_mids}).with_columns(pl.col('phi').cut(bins).alias('bin'))
        df        = full_bins.join(binned, on = 'bin', how = 'left').sort('phi')
        
        scaled_impact_binned = df['scaled impact (binned)']
        if len(scaled_impact_binned) < 10:
            return pl.DataFrame()
            
        scaled_impact_binned = pl.concat([pl.Series([0.0]), scaled_impact_binned])
        phi                  = np.insert(bin_mids, 0, 0)
        
        def execution_profile_model(phi, exponent, Y):
            return  Y * (phi ** exponent)
        
        phi    = np.array(phi)
        impact = np.array(scaled_impact_binned)
        
        mask       = (phi > 0) & np.isfinite(impact)
        phi_fit    = phi[mask]
        impact_fit = impact[mask]
    
        if len(phi_fit) < 10 or len(impact_fit) < 10:
            return pl.DataFrame()
        
        try:
            params, covariance = curve_fit(execution_profile_model, phi_fit, impact_fit, p0 = [0.5, 1], maxfev = 100000)
    
        except RuntimeError:
            return pl.DataFrame()
            
        phi_est = params[0]
        
        n   = len(impact_fit)
        p   = len(params)          # number of parameters (2 here)
        dof = n - p                # degrees of freedom
        
        alpha = 0.05               # 95% CI
        tval  = t.ppf(1 - alpha / 2, dof)
        se    = np.sqrt(np.diag(covariance))
        phi_interval_half_width = tval * se[0]    
        phi_var = covariance[0, 0]
        
        ###
        
        ### Decay profile ###
        
        bins = np.linspace(1, 2, 51)
        bin_mids = (bins[:-1] + bins[1:]) / 2
        
        df        = decay_data.with_columns(pl.col('z').cut(bins).alias('bin'))
        binned    = df.group_by('bin').agg(pl.col('scaled impact').mean().alias('scaled impact (binned)')).sort('bin')
        full_bins = pl.DataFrame({'z': bin_mids}).with_columns(pl.col('z').cut(bins).alias('bin'))
        df        = full_bins.join(binned, on = 'bin', how = 'left').sort('z')
       
        z         = np.array(bin_mids)
        impact    = np.array(df['scaled impact (binned)'])
        
        def decay_model(z, beta, Y):
            return  Y * (z ** (1 - beta) - (z - 1) ** (1 - beta))
        
        mask       = (z > 1) & np.isfinite(impact)
        z_fit      = z[mask]
        impact_fit = impact[mask]
    
        if len(z_fit) < 10 or len(impact_fit) < 10:
            return pl.DataFrame()
    
        try:
            params, covariance = curve_fit(decay_model, z_fit, impact_fit, p0 = [0.2, 1], maxfev = 100000)
    
        except RuntimeError:
            return pl.DataFrame()    
        
        beta_est = params[0]
    
        n   = len(z_fit)
        p   = len(params)          # number of parameters (2 here)
        dof = n - p                # degrees of freedom
        
        alpha = 0.05               # 95% CI
        tval  = t.ppf(1 - alpha / 2, dof)
        se    = np.sqrt(np.diag(covariance))
        beta_interval_half_width = tval * se[0]
        beta_var = covariance[0, 0] 
        
        L     = extract_ST_run_lengths(df_year, group = 'year', N = N, trader_distribution = participation_method,
                                       alpha = alpha_tp)
        signs = df_year['Trade Sign']
        
        if len(L) == 0:
            continue
    
        L     = pl.DataFrame({'L' : L})
        signs = pl.DataFrame({'Trade Sign' : signs})
        
        taus, C = acf_fft(signs['Trade Sign'], max_lag = 100000)
        out     = estimate_gamma(taus, C, min_points = 100, nbins = 100, trade_signs = signs['Trade Sign'])
        
        alpha, xmin = estimate_alpha(L['L'])
        
        years_col.append(year)
        ticker_col.append(ticker)
        xmin_col.append(xmin)
        alpha_col.append(alpha)
        gamma_nlls_col.append(out['gamma (nlls)'])
        gamma_psd_col.append(out['gamma (PSD)'])
        sql_est_col.append(exp_est)
        sql_var_col.append(exp_var)
        sql_interval_half_width_col.append(exp_interval_half_width)
        phi_est_col.append(phi_est)
        phi_var_col.append(phi_var)
        phi_interval_half_width_col.append(phi_interval_half_width)
        beta_est_col.append(beta_est)
        beta_var_col.append(beta_var)
        beta_interval_half_width_col.append(beta_interval_half_width)

    data_lists = [years_col, ticker_col, xmin_col, alpha_col, gamma_nlls_col, gamma_psd_col, 
                  phi_est_col, phi_interval_half_width_col, beta_est_col, beta_interval_half_width_col]
    
    if any(not x for x in data_lists):
        return pl.DataFrame()
        
    alpha_gamma_df = pl.DataFrame({
            'year'                    : years_col,
            'ticker'                  : ticker_col,
            'N'                       : N,
            #'delta'                   : alpha_tp,
            'xmin'                    : xmin_col,
            'alpha'                   : alpha_col,
            'gamma (nlls)'            : gamma_nlls_col,
            'gamma (psd)'             : gamma_psd_col,
            'impact curve exponent'   : sql_est_col,
            'impact curve exponent var' : sql_var_col,
            'impact curve half width' : sql_interval_half_width_col,
            'phi exponent'            : phi_est_col, 
            'phi exponent var'        : phi_var_col,
            'phi exponent half width' : phi_interval_half_width_col,
            'beta'                    : beta_est_col,
            'beta var'                : beta_var_col,
            'beta half width'         : beta_interval_half_width_col}, 
             schema = {
            'year'                    : pl.Int64,
            'ticker'                  : pl.Utf8,
            'N'                       : pl.Int64,
            #'delta'                   : pl.Float64,
            'xmin'                    : pl.Float64,
            'alpha'                   : pl.Float64,
            'gamma (nlls)'            : pl.Float64,
            'gamma (psd)'             : pl.Float64,
            'impact curve exponent'   : pl.Float64,
            'impact curve exponent var' : pl.Float64,
            'impact curve half width' : pl.Float64,
            'phi exponent'            : pl.Float64,
            'phi exponent var'        : pl.Float64,
            'phi exponent half width' : pl.Float64,
            'beta'                    : pl.Float64,
            'beta var'                : pl.Float64,
            'beta half width'         : pl.Float64})
    
    return alpha_gamma_df
    

In [5]:
#param_grid = {'N': [50, 100, 500, 1000, 1500], 'alpha_tp': [1.5, 2, 3, 4, 5]}
#param_grid = {'N': [5, 10, 20, 30, 40, 50, 100, 500, 1000, 1500], 'alpha_tp': [1.5, 2, 3, 4, 5]}
param_grid = {'N': [5, 10, 20, 30, 40, 50, 100, 500, 1000, 1500]}
expanded_grid = list(ParameterGrid(param_grid))
print(expanded_grid)

[{'N': 5}, {'N': 10}, {'N': 20}, {'N': 30}, {'N': 40}, {'N': 50}, {'N': 100}, {'N': 500}, {'N': 1000}, {'N': 1500}]


In [6]:
b2.get_file('top_250.csv')
top_250 = pd.read_csv('top_250.csv')

top_250_tickers = top_250['Ticker'].tolist()

In [7]:
%%time
files = b2.list_files(path = 'data/trades/top_250')

for f in files:
    b2.get_file(f'data/trades/top_250/{f}')

def load_stock(parquet):
    try:
        df = pl.read_parquet(parquet)
        return df.sort(['DateTime', 'ExchangeSequenceNo'])
    except FileNotFoundError:
        return None

exchange = 'XJSE'
stocks = {
    ticker: df
    for ticker in top_250_tickers
    if (df := load_stock(f'{ticker}_{exchange}.parquet')) is not None
}

CPU times: user 1min 7s, sys: 25 s, total: 1min 32s
Wall time: 1min 35s


In [8]:
#process_stock_pl(stocks['GRT'], group = 'year', N = 10, participation_method = 'homogenous', alpha = 2)

In [16]:
%%time
metaorders_stylised_facts(stocks['GRT'], 100, 2)

CPU times: user 1min 36s, sys: 14.4 s, total: 1min 50s
Wall time: 1min 19s


year,ticker,N,delta,xmin,alpha,gamma (nlls),gamma (psd),impact curve exponent,impact curve exponent var,impact curve half width,phi exponent,phi exponent var,phi exponent half width,beta,beta var,beta half width
i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2023,"""GRT""",100,2.0,3.0,1.905225,0.500202,0.482646,0.074267,0.000757,0.055733,0.869865,0.001025,0.064369,0.217672,0.000004,0.004168
2024,"""GRT""",100,2.0,3.0,1.836304,0.340167,0.433737,29.685004,1.2975e9,72984.261235,0.739666,0.001375,0.074551,0.141446,0.000008,0.005569
2025,"""GRT""",100,2.0,3.0,1.919214,0.393937,0.489648,0.127933,0.000832,0.058452,0.723623,0.001234,0.070629,0.141869,0.000021,0.009194


In [ ]:
top_250_tickers

In [8]:
participation_method = 'homogenous'

b2.create_folder('data', ensure_exists = True)
b2.create_folder('data/grid_search_6', ensure_exists = True)
b2.create_folder(f'data/grid_search_6/{exchange}', ensure_exists = True)
b2.create_folder(f'data/grid_search_6/{exchange}/{participation_method}', ensure_exists = True)

In [9]:
%time

start_from = 'CMH'
start = False

counter = 155
for ticker, stock_data in stocks.items():

    if ticker == start_from:
        start = True

    if not start:
        continue
        
    print(f'{counter}: {ticker}')
    counter += 1

    results_list = []
    for params in expanded_grid:
        
        N        = params['N']
        #alpha_tp = params['alpha_tp']
        print(f'{N}')
    
        res = metaorders_stylised_facts(stock_data, N, alpha_tp = 2, participation_method = participation_method)
    
        if not res.is_empty():
            results_list.append(res)

    if results_list:
        results_df = pl.concat(results_list, how = 'vertical', rechunk = True)
        results_df.write_parquet(f'{ticker}_{exchange}_{participation_method}_grid_search.parquet')
        b2.put_file(f'{ticker}_{exchange}_{participation_method}_grid_search.parquet', f'data/grid_search_6/{exchange}/{participation_method}')
    else:
        results_df = pl.DataFrame()

CPU times: user 10 µs, sys: 0 ns, total: 10 µs
Wall time: 21 µs
155: CMH
5
10
20
30
40
50
100
500
1000
1500
156: MDI
5
10
20
30
40
50
100
500
1000
1500
157: JBL
5
10
20
30
40
50
100
500
1000
1500
158: OAO
5
10
20
30
40
50
100
500
1000
1500
159: CLH
5
10
20
30
40
50
100
500
1000
1500
160: SDL
5
10
20
30
40
50
100
500
1000
1500
161: ZZD
5
10
20
30
40
50
100
500
1000
1500
162: YYLBEE
5
10
20
30
40
50
100
500
1000
1500
163: CTA
5
10
20
30
40
50
100
500
1000
1500
164: QFH
5
10
20
30
40
50
100
500
1000
1500
165: GML
5
10
20
30
40
50
100
500
1000
1500
166: ZED
5
10
20
30
40
50
100
500
1000
1500
167: SCD
5
10
20
30
40
50
100
500
1000
1500
168: NVS
5
10
20
30
40
50
100
500
1000
1500
169: ART
5
10
20
30
40
50
100
500
1000
1500
170: BWN
5
10
20
30
40
50
100
500
1000
1500
171: OAS
5
10
20
30
40
50
100
500
1000
1500
172: APF
5
10
20
30
40
50
100
500
1000
1500
173: MMP
5
10
20
30
40
50
100
500
1000
1500
174: EPS
5
10
20
30
40
50
100
500
1000
1500
175: EMN
5
10
20
30
40
50
100
500
1000
1500
176: MCZ


/tmp/ipykernel_9764/4169924894.py:179: OptimizeWarning:

Covariance of the parameters could not be estimated



40
50
100
500
1000
1500
191: MTA
5
10
20
30
40
50
100
500
1000
1500
192: WEZ
5
10
20
30
40
50
100
500
1000
1500
193: PBG
5
10
20
30
40
50
100
500
1000
1500
194: ENX
5
10
20
30
40
50
100
500
1000
1500
195: HLM
5
10
20
30
40
50
100
500
1000
1500
196: ACT
5
10
20
30
40
50
100
500
1000
1500
197: RMH
5
10
20
30
40
50
100
500
1000
1500
198: ADR
5
10
20
30
40
50
100
500
1000
1500
199: FGL
5
10
20
30
40
50
100
500
1000
1500
200: NWL
5
10
20
30
40
50
100
500
1000
1500
201: AEG
5
10
20
30
40
50
100
500
1000
1500
202: CGR
5
10
20
30
40
50
100
500
1000
1500
203: SEP
5
10
20
30
40


/tmp/ipykernel_9764/4169924894.py:179: OptimizeWarning:

Covariance of the parameters could not be estimated



50


/tmp/ipykernel_9764/4169924894.py:179: OptimizeWarning:

Covariance of the parameters could not be estimated



100
500
1000
1500
204: NRL
5
10
20
30
40
50
100
500
1000
1500
205: SOLBE1
5
10
20
30
40
50
100
500
1000
1500
206: ISA
5
10
20
30
40
50
100
500
1000
1500
207: VUN
5
10
20
30
40
50
100
500
1000
1500
208: 4SI
5
10
20
30
40
50
100
500
1000
1500
209: TTO
5
10
20
30
40
50
100
500
1000
1500
210: AME
5
10
20
30
40
50
100
500
1000
1500
211: PMV
5
10
20
30
40
50
100
500
1000
1500
212: CKS
5
10
20
30
40
50
100
500
1000
1500
213: RTN
5
10
20
30
40
50
100
500
1000
1500
214: PPR
5
10
20
30
40
50
100
500
1000
1500
215: HUG
5
10
20
30
40
50
100
500
1000
1500
216: DLT
5
10
20
30
40
50
100
500
1000
1500
217: TMT
5
10
20
30
40
50
100
500
1000
1500
218: ISB
5
10
20
30
40
50
100
500
1000
1500
219: BRT
5
10
20
30
40
50
100
500
1000
1500
220: SOH
5
10
20
30
40
50
100
500
1000
1500
221: CAC
5
10
20
30
40
50
100
500
1000
1500
222: EMH
5
10
20
30
40
50
100
500
1000
1500
223: KBO
5
10
20
30
40
50
100
500
1000
1500
224: AON
5
10
20
30
40
50
100
500
1000
1500
225: TRL
5
10
20
30
40
50
100
500
1000
1500
226: NCS
5
